# Paired Switching: Research Notebook

This notebook is used to test and validate the core logic of the `PairedSwitching` algorithm outside of the backtesting environment. This allows for rapid iteration and debugging of the data collection, clustering, and analysis functions.

**Workflow:**
1.  **Universe Selection:** Replicate the coarse and fine universe filtering.
2.  **Data Fetching:** Load 3 years of historical data for the selected universe.
3.  **Metric Collection:** Apply the `collect_stock_metrics` function.
4.  **Clustering:** Run the `perform_clustering` function on the collected metrics.
5.  **Visualization:** Plot the clusters to visually inspect the results.

In [1]:
# Cell 1: Imports and Initialization
from AlgorithmImports import * # Make sure we have the QC libraries
from QuantConnect.Configuration import Config
from QuantConnect.DataSource import *
from datetime import datetime, timedelta

# Explicitly set configuration to ensure API data is used correctly
#Config.Set("data-provider", "QuantConnect.Api.ApiDataProvider")
#Config.Set("job-user-id", "457946")
#Config.Set("api-access-token", "eceb9dbf91626341b043e1e6720f220f50ff60e7598cee557960ab047f02ba55")
#Config.Set("data-folder", "data")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns




qb = QuantBook()

In [ ]:
# 1. Configuration
target_date = datetime(2023, 10, 18)
qb.SetStartDate(target_date)

# 2. Global Universe Fetch
# Using FundamentalUniverse class directly bypasses scope issues.
universe_history = qb.UniverseHistory(FundamentalUniverse, target_date - timedelta(days=1), target_date)

coarse_symbols = []
if universe_history is not None:
    for collection in universe_history:
        raw_list = [x for x in collection]
        
        # Ensure we are looking at the full market, not just a few added stocks
        if len(raw_list) < 100:
            continue 
            
        print(f"✅ Success! Found {len(raw_list)} market records.")
        
        # 4. Coarse Filter: Price > 1 and must have fundamental data
        filtered = [x for x in raw_list if x.Price > 1 and x.HasFundamentalData]
        sorted_list = sorted(filtered, key=lambda x: x.DollarVolume, reverse=True)
        coarse_symbols = [x.Symbol for x in sorted_list[:500]]
        
        print(f"Selected {len(coarse_symbols)} symbols via Coarse filter.")
        break

# 5. Fine Fundamental Data Fetch
final_symbols = []
if coarse_symbols:
    print("Fetching Fine fundamental data for candidates...")
    # Requesting just MarketCap for the 1000 candidates
    fine_data = qb.GetFundamental(coarse_symbols, "ValuationRatios.MarketCap", target_date, target_date)

    if not fine_data.empty:
        # Get the latest row of data
        latest_row = fine_data.iloc[-1]
        
        # Convert to numeric to handle 'object' dtypes
        mcap_series = pd.to_numeric(latest_row, errors='coerce').dropna()
        
        # KEY CHANGE: Instead of splitting strings, we match the index directly to our objects.
        # QuantConnect's GetFundamental index level often contains the Symbol objects.
        # We sort them and take the top 100.
        top_500_series = mcap_series.nlargest(100)
        
        # Extract the index. If they are already Symbols, we are done.
        # If they are strings, we map them back using a dictionary.
        potential_symbols = top_500_series.index.tolist()
        
        if len(potential_symbols) > 0:
            if isinstance(potential_symbols[0].Value, str):
                # If the index is strings (e.g., 'AAPL.ValuationRatios.MarketCap')
                symbol_lookup = {s.Value: s for s in coarse_symbols}
                for item in potential_symbols:
                    print(f" ticking down the list: {item}")
                    ticker = str(item).split('.')[0]
                    if ticker in symbol_lookup:
                        final_symbols.append(symbol_lookup[ticker])
            else:
                # If the index is already Symbol objects
                print(f" Nothing : {item}")
                final_symbols = potential_symbols
                

        print(f"✅ Fine selection complete: {len(final_symbols)} symbols selected.")
    else:
        print("⚠️ Fine data fetch failed. Falling back to top 100 coarse symbols.")
        final_symbols = coarse_symbols[:100]

# 6. Final History Download
if final_symbols:
    print(f"Downloading 3-year history for {len(final_symbols)} symbols...")
    price_history = qb.History(final_symbols, target_date - timedelta(days=3*365), target_date, Resolution.Daily)
    
    if not price_history.empty:
        print(f" Success! Price History Shape: {price_history.shape}")
    else:
        print(" History call returned no data. Check your symbol objects or resolution.")
else:
    print(" No symbols available for history download.")

In [ ]:
from datetime import datetime

# Test: Get Market Cap for SPY on our target date
# Syntax: qb.GetFundamental(tickers, selector, start_date, end_date)
try:
    test_fundamental = qb.GetFundamental("SPY", "ValuationRatios.MarketCap", datetime(2023, 10, 18), datetime(2023, 10, 18))
    
    if not test_fundamental.empty:
        print("✅ SUCCESS: Fundamental data is accessible.")
        print(f"Market Cap for SPY: {test_fundamental.iloc[-1]}")
    else:
        print("❌ EMPTY: The call succeeded, but returned no data.")
        print("This usually means the 'US Equity Coarse Universe' subscription is NOT active.")
except Exception as e:
    print(f"⚠️ ERROR: Method call failed. {e}")

In [ ]:
# Cell 3: Ported `collect_stock_metrics` function

def collect_stock_metrics(history_df, symbols):
    """Collect price, momentum, and other metrics for all stocks in universe."""
    if history_df.empty:
        print("No historical data provided")
        return None
    
    metrics = {}
    
    # Create mapping for Ticker -> Symbol to handle potential string keys in history
    ticker_to_symbol = {str(s.Value): s for s in symbols}
    
    # level=0 is the Symbol index
    for identifier, sym_history in history_df.groupby(level=0):
        
        # Resolve the identifier to a Symbol object
        symbol_obj = None
        if isinstance(identifier, Symbol):
            symbol_obj = identifier
        elif isinstance(identifier, str):
            symbol_obj = ticker_to_symbol.get(identifier)
        
        if symbol_obj is None:
            continue
        
        # We need at least a year's worth of data to match the algorithm's logic
        if len(sym_history) < 252:
            continue
        
        # Take the last 252 days of data for calculation
        sym_history = sym_history.iloc[-252:]

        # Extract data safely
        if 'close' not in sym_history.columns:
            continue
            
        close_prices = np.array(sym_history['close'].values, dtype=np.float64)
        
        # Handle volume if present
        if 'volume' in sym_history.columns:
            volumes = np.array(sym_history['volume'].values, dtype=np.float64)
            avg_volume = np.mean(volumes)
        else:
            avg_volume = 0
        
        # Calculate metrics
        current_price = close_prices[-1]
        
        # Price Change (over the 252 day period)
        start_price = close_prices[0]
        price_change = (current_price - start_price) / start_price if start_price != 0 else 0
        
        # Momentum (21 days)
        if len(close_prices) > 21:
            prev_price = close_prices[-21]
            momentum = (current_price - prev_price) / prev_price if prev_price != 0 else 0
        else:
            momentum = 0
        
        # Volatility (over the 252 day period)
        if len(close_prices) > 1:
            prices_t = close_prices[1:]
            prices_t_minus_1 = close_prices[:-1]
            
            with np.errstate(divide='ignore', invalid='ignore'):
                returns = (prices_t - prices_t_minus_1) / prices_t_minus_1
            
            returns = returns[np.isfinite(returns)]
            
            if len(returns) > 0:
                volatility = np.std(returns)
            else:
                volatility = 0
        else:
            volatility = 0
        
        direction = 1 if price_change > 0 else -1
        
        metrics[symbol_obj] = {
            'price': current_price,
            'price_change': price_change,
            'momentum': momentum,
            'volatility': volatility,
            'volume': avg_volume,
            'direction': direction
        }
    
    print(f"Collected metrics for {len(metrics)} stocks")
    
    if len(metrics) > 0:
        return pd.DataFrame(metrics).T
    else:
        return None

In [4]:
# Cell 4: Ported `perform_clustering` function

def perform_clustering(metrics_df):
    """Classify stocks into 6 Market Regimes based on Momentum and Volatility."""
    if metrics_df is None or len(metrics_df) < 1:
        print(f"Insufficient data for clustering (need 1+, have {len(metrics_df) if metrics_df is not None else 0})")
        return None, None
    
    regime_names = [
        "Calm Bull", "Volatile Bull",
        "Calm Bear", "Volatile Bear",
        "Calm Sideways", "Volatile Sideways"
    ]
    groups = {name: [] for name in regime_names}
    group_assignments = {} # To hold symbol -> group_name mapping
    
    for symbol, row in metrics_df.iterrows():
        mom = row['momentum']
        vol = row['volatility']
        
        # Determine Trend
        if mom > 0.02: trend = "Bull"
        elif mom < -0.02: trend = "Bear"
        else: trend = "Sideways"
        
        # Determine Volatility (1.5% daily threshold)
        if vol > 0.015: vol_type = "Volatile"
        else: vol_type = "Calm"
        
        # Assign Group ID
        grp = f"{vol_type} {trend}"
        
        groups[grp].append(symbol)
        group_assignments[symbol] = grp
            
    print("Clustering complete.")
    return groups, group_assignments

In [ ]:
# Cell 5: Execution and Results

# 1. Collect metrics
metrics_df = collect_stock_metrics(price_history, final_symbols)

# 2. Perform clustering
if metrics_df is not None:
    correlation_groups, group_assignments = perform_clustering(metrics_df)
    
    # Add cluster assignments to the dataframe for plotting
    metrics_df['cluster'] = metrics_df.index.map(group_assignments)
    
    print("\n--- METRICS DATAFRAME HEAD ---")
    print(metrics_df.head())
    
    print("\n--- CLUSTER SIZES ---")
    for name, symbols in sorted(correlation_groups.items()):
        print(f"{name:<20}: {len(symbols)} members")
else:
    print("Metrics DataFrame could not be generated.")

In [ ]:
# Cell 6: Visualization

if metrics_df is not None and 'cluster' in metrics_df.columns:
    plt.figure(figsize=(12, 8))
    sns.scatterplot(data=metrics_df, x='volatility', y='momentum', 
                    hue='cluster', palette='viridis', s=50, alpha=0.7)
    
    # Add lines for the regime boundaries
    plt.axhline(0.02, color='grey', linestyle='--', linewidth=1)
    plt.axhline(-0.02, color='grey', linestyle='--', linewidth=1)
    plt.axvline(0.015, color='grey', linestyle='--', linewidth=1)
    
    plt.title(f'Clustering: Momentum vs Volatility')
    plt.xlabel('Volatility (Std Dev of Daily Returns)')
    plt.ylabel('21-Day Momentum')
    plt.legend(title='Cluster')
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.show()